# 16 — Aging as a dynamical system

Earlier phases compared young with old. Here age is time, and trajectories are modelled with differential equations. The bulk atlas (14) has **10 ages from 1 to 27 months**, so trajectories have enough points to fit; the single-cell atlas adds per-mouse cell states. All models live in `src/dynamics.py`, with settings fixed before running.

| Part | Question | Model | Data |
| --- | --- | --- | --- |
| 7a | Does senescence-like damage **accelerate** with age? | Karin et al. 2019: $dX/dt = \eta t - \beta X/(\kappa + X)$, vs linear and exponential | bulk p16 (*Cdkn2a*), and the MOFA aging factor of 14 |
| 7b | Do cells of one type become more **heterogeneous** with age, and is there any sign of a critical transition? | variance, leading eigenvalue, correlation within each mouse | single-cell robust rows, depth-matched; bulk spread between mice |
| 7c | How fast does the **thymus** stop making naive T cells? Can 5 ages even tell? | thymic output → naive → memory compartments, profile likelihood | single-cell spleen and lung T cells |
| 7e | Can a dynamical model **forecast an age it never saw**? | nearest age, interpolation, spline, sigmoid, linear ODE, SINDy, neural ODE; optimal transport for single cells | bulk programs, 17 organs; single-cell distributions |

**Status of this notebook: exploratory.** Every model here is fitted to 10 bulk ages or 4–6 single-cell ages, with age tied to batch in the single-cell data. Results are reported as what the data can and cannot distinguish, not as discoveries; a model that wins here wins on this data, not in general.

### 1. Setup

In [ ]:
import sys, io, contextlib
sys.path.append("..")          # so "src" can be imported from the notebook folder

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
from scipy.stats import spearmanr
from scipy.integrate import solve_ivp
from src import aging as ag
from src import dynamics as D

A = ag.paths("atlas")
atlas = pd.read_csv(A.figure("atlas.csv"))
atlas["robust"] = atlas["verdict"].eq("age signal") & ~atlas["tech_confounded"].astype(bool)

bulk = ad.read_h5ad(ag.ROOT / "data" / "external" / "190304_maca_bulk.h5ad")
bulk = bulk[bulk.obs["Organ"].astype(str).ne("nan").to_numpy()]
lib = np.asarray(bulk.X.sum(axis=1)).ravel()
bulk = bulk[lib > 1e6].copy()
sc.pp.normalize_total(bulk, target_sum=1e6)
sc.pp.log1p(bulk)
bobs = bulk.obs[["Organ", "MouseID", "Age", "Sex", "Plate"]].astype({"Organ": str, "MouseID": str})

### 2. (7a) Accumulation: linear, exponential, or saturating removal?

Karin et al. 2019 model senescent cells as produced at a rate that grows with age ($\eta t$) and removed by a process that **saturates** ($\beta X/(\kappa + X)$). Once production outruns the maximal removal rate, $X$ accelerates. Fitted per organ to p16 (*Cdkn2a*, log CPM, one point per mouse), and to the MOFA aging factor (F2 of 14, one point per mouse), against a straight line and an exponential. Lower AIC wins; a difference under 2 is a tie. On simulated Karin data the exponential ties with Karin too, so a Karin win needs a clear margin.

In [ ]:
p16 = pd.Series(np.asarray(bulk[:, "Cdkn2a"].X.toarray() if hasattr(bulk.X, "toarray") else bulk[:, "Cdkn2a"].X).ravel(),
                index=bulk.obs_names)
rows = []
for organ, idx in bobs.groupby("Organ").groups.items():
    y = p16.loc[idx]
    if (y > 0).mean() < 0.5:                       # p16 must be detected in most samples
        continue
    fit = D.fit_growth(bobs.loc[idx, "Age"].to_numpy(), y.to_numpy())
    aic = {k: v["aic"] for k, v in fit.items()}
    rows.append({"series": f"p16, {organ}", **aic, "best": min(aic, key=aic.get),
                 "karin_margin": min(aic["linear"], aic["exponential"]) - aic["karin"]})

with h5py.File(A.figure("mofa_bulk.hdf5")) as f:
    Z = pd.DataFrame(f["expectations/Z/all"][:].T, index=[s.decode() for s in f["samples/all"][:]])
F2 = Z[1]
mouse_age = bobs.drop_duplicates("MouseID").set_index("MouseID")["Age"].loc[F2.index]
sign = np.sign(spearmanr(F2, mouse_age)[0])
fit = D.fit_growth(mouse_age.to_numpy(), (sign * F2).to_numpy())
aic = {k: v["aic"] for k, v in fit.items()}
rows.append({"series": "MOFA aging factor (all organs)", **aic, "best": min(aic, key=aic.get),
             "karin_margin": min(aic["linear"], aic["exponential"]) - aic["karin"]})
growth = pd.DataFrame(rows)
growth.round(2).to_csv(A.figure("dynamics_growth_models.csv"), index=False)
print(growth["best"].value_counts().to_dict(), "| Karin ahead by > 2 AIC:", (growth["karin_margin"] > 2).sum(), "of", len(growth))
growth.round(1)

### 3. (7a, continued) The spread between mice

The stochastic Karin model predicts that individuals **drift apart** with age: the variance of $X$ between individuals grows. For p16 and the aging factor: standard deviation between mice at each age, and its Spearman ρ with age. A simulation of the model shows the expected shape.

This prediction is **not specific** to the Karin model: almost any model in which individuals differ in their rates predicts growing spread, and so does any measure whose variance grows with its mean. A rise is consistent with the model but cannot favour it over alternatives.

In [ ]:
sd_rows = []
for organ, idx in bobs.groupby("Organ").groups.items():
    d = pd.DataFrame({"age": bobs.loc[idx, "Age"], "y": p16.loc[idx]})
    if (d["y"] > 0).mean() < 0.5:
        continue
    s = d.groupby("age")["y"].std()
    sd_rows.append({"series": f"p16, {organ}", "rho_sd_age": spearmanr(s.index, s.values)[0]})
s = pd.DataFrame({"age": mouse_age, "y": sign * F2}).groupby("age")["y"].std()
sd_rows.append({"series": "MOFA aging factor", "rho_sd_age": spearmanr(s.index, s.values)[0]})
spread = pd.DataFrame(sd_rows)
print("median rho(SD, age):", round(spread["rho_sd_age"].median(), 2), "| positive:", (spread["rho_sd_age"] > 0).sum(), "of", len(spread))

ts, xs = D.karin_sde(eta=0.02, beta=0.5, kappa=1.0, eps=0.002, t_end=30, n=2000)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for q in [5, 25, 50, 75, 95]:
    axes[0].plot(ts, np.percentile(xs, q, axis=0), color="k", lw=0.8 if q != 50 else 2)
axes[0].set_xlabel("age (months)"); axes[0].set_ylabel("X (simulated)"); axes[0].set_title("Karin model: individuals drift apart")
sns.barplot(data=spread.sort_values("rho_sd_age"), x="rho_sd_age", y="series", ax=axes[1], color="gray")
axes[1].set_xlabel("Spearman rho of between-mouse SD with age"); axes[1].set_ylabel("")
fig.savefig(A.figure("dynamics_karin.png"), dpi=200, bbox_inches="tight")
spread.round(2)

### 4. (7b) Heterogeneity with age, and early-warning proxies

Near a tipping point a system recovers slowly from perturbations ("critical slowing down"), so its fluctuations grow and become correlated. In a cell population that shows as: more **heterogeneity** between cells of one type, a larger share of variance on the **leading axis** (one dominant mode), and stronger **gene–gene correlation**. Per robust row and mouse (≥ 30 cells, at most 200): every cell is first **downsampled to 1,000 counts**, because sequencing depth alone changes all three measures. Then, on that row's 500 most variable genes:

- `heterogeneity`: mean distance of cells to their mouse's centroid
- `leading_eig`: fraction of variance on the first principal axis
- `mean_abs_corr`: mean absolute correlation between genes

Trend with age over mice (Spearman ρ); `stable` = the sign holds with any one age group left out.

**What this can and cannot test.** Critical slowing down is defined for one system observed repeatedly as it approaches a transition. Here each mouse is seen once, and each age group is a different set of mice processed in its own batch (the 21m mice are all female). So these are cross-sectional proxies: a rise in heterogeneity describes aged cell populations and could also come from batch, and the absence of a rising leading eigenvalue is weak evidence against an approaching transition, not a test of one.

In [ ]:
def ews(Xc):
    Xc = Xc - Xc.mean(axis=0)
    het = np.sqrt((Xc ** 2).sum(axis=1)).mean()
    ev = np.linalg.eigvalsh(np.cov(Xc, rowvar=False))
    C = np.corrcoef(Xc, rowvar=False)
    C = C[np.triu_indices_from(C, 1)]
    return het, ev[-1] / ev.sum(), np.nanmean(np.abs(C))

rows = []
for tissue in sorted(atlas.loc[atlas["robust"], "tissue"].unique()):
    a = sc.read_h5ad(ag.paths(tissue).processed)
    for ct in atlas.loc[atlas["robust"] & atlas["tissue"].eq(tissue), "cell_type"]:
        sub = a[(a.obs["cell_ontology_class"] == ct).to_numpy() & a.obs["age"].isin(ag.YOUNG + ag.OLD).to_numpy()]
        drop = ag.main_kw(ct, tissue).get("drop_mice", [])
        sub = sub[~sub.obs["mouse.id"].isin(drop).to_numpy()]
        cnt = sub.layers["counts"]
        tot = np.asarray(cnt.sum(axis=1)).ravel()
        sub = sub[tot >= 1000].copy()
        sub.X = sub.layers["counts"].copy()
        sc.pp.downsample_counts(sub, counts_per_cell=1000, random_state=0)
        sc.pp.log1p(sub)
        with contextlib.redirect_stderr(io.StringIO()):
            sc.pp.highly_variable_genes(sub, n_top_genes=500, flavor="seurat")
        Xs = sub[:, sub.var["highly_variable"].to_numpy()].X
        Xs = Xs.toarray() if hasattr(Xs, "toarray") else np.asarray(Xs)
        mice = sub.obs["mouse.id"].astype(str).to_numpy()
        rng = np.random.default_rng(0)
        for m in np.unique(mice):
            idx = np.where(mice == m)[0]
            if len(idx) < 30:
                continue
            idx = rng.choice(idx, min(len(idx), 200), replace=False)
            h, l, c = ews(Xs[idx])
            rows.append({"tissue": tissue, "cell_type": ct, "mouse": m, "age": sub.obs["age"].astype(str).iloc[idx[0]],
                         "heterogeneity": h, "leading_eig": l, "mean_abs_corr": c})
    del a
ews_mouse = pd.DataFrame(rows)
ews_mouse["months"] = ews_mouse["age"].map(ag.AGE_MONTHS)
ews_mouse.round(4).to_csv(A.figure("dynamics_ews_by_mouse.csv"), index=False)

def trend(d, col):
    rho = spearmanr(d["months"], d[col])[0]
    loo = [spearmanr(x["months"], x[col])[0] for a in d["age"].unique() if (x := d[d["age"] != a])["months"].nunique() > 1]
    return pd.Series({"rho": rho, "stable": all(np.sign(r) == np.sign(rho) for r in loo), "mice": len(d)})

ews_trend = pd.concat({col: ews_mouse.groupby(["tissue", "cell_type"]).apply(trend, col=col, include_groups=False)
                       for col in ["heterogeneity", "leading_eig", "mean_abs_corr"]}, names=["measure"]).reset_index()
ews_trend.round(3).to_csv(A.figure("dynamics_ews_trend.csv"), index=False)
ews_trend.groupby("measure").agg(rows=("rho", "size"), rising=("rho", lambda v: (v > 0).sum()),
                                 rising_stable=("rho", lambda v: ((v > 0) & ews_trend.loc[v.index, "stable"]).sum()),
                                 median_rho=("rho", "median")).round(2)

### 5. (7c) Thymic involution: naive and memory T cells

The thymus shrinks with age, so fewer naive T cells are made and the pool shifts toward memory cells. Each T cell is called naive or memory by scoring naive genes (*Sell*, *Ccr7*, *Lef1*, *Tcf7*) against memory/effector genes (*Cd44*, *S100a4*, *Ccl5*, *Itgb1*); the naive fraction per mouse is the data.

Only fractions are observed, so the total T-cell pool is taken as held steady (homeostasis) and the naive fraction $f$ follows
$$\frac{df}{dt} = \theta e^{-\delta t} - k f, \qquad f(0) = f_0,$$
thymic input $\theta$ that declines at the involution rate $\delta$, against naive loss $k$; $f_0$ is set from the youngest mice. (A first version tracked naive and memory pools separately; fitted to fractions alone it drove both pools to zero and their ratio to numerical noise, giving a curve below zero. It was replaced before any result was written up.)

A **profile likelihood** for $\delta$ asks whether the data pin it down: the fit is repeated with $\delta$ fixed on a grid from 0.003 to 3 per month and compared with the best fit found anywhere. A ratio near 1 over a wide range means these ages cannot tell those involution rates apart.

In [ ]:
NAIVE, MEM = ["Sell", "Ccr7", "Lef1", "Tcf7"], ["Cd44", "S100a4", "Ccl5", "Itgb1"]
tc = []
for tissue, cts in [("Spleen", ["T cell"]), ("Lung", ["CD4-positive, alpha-beta T cell", "CD8-positive, alpha-beta T cell"])]:
    a = sc.read_h5ad(ag.paths(tissue).processed)
    t = a[a.obs["cell_ontology_class"].isin(cts).to_numpy()].copy()
    sc.tl.score_genes(t, [g for g in NAIVE if g in t.var_names], score_name="naive", random_state=0)
    sc.tl.score_genes(t, [g for g in MEM if g in t.var_names], score_name="memory", random_state=0)
    t.obs["is_naive"] = t.obs["naive"] > t.obs["memory"]
    f = t.obs.groupby("mouse.id", observed=True).agg(age=("age", "first"), cells=("is_naive", "size"), naive_frac=("is_naive", "mean"))
    tc.append(f[f["cells"] >= 20].reset_index().assign(tissue=tissue))
    del a
tc = pd.concat(tc)
tc["months"] = tc["age"].astype(str).map(ag.AGE_MONTHS)
print(tc.groupby(["tissue", "months"])["naive_frac"].agg(["size", "mean"]).round(2))

In [ ]:
prof_rows, fits = [], {}
for tissue, d in tc.groupby("tissue"):
    t, f = d["months"].to_numpy(float), d["naive_frac"].to_numpy(float)
    f0 = d.loc[d["months"] == d["months"].min(), "naive_frac"].mean()
    full = D.fit_tcell(t, f, f0)
    fits[tissue] = (full, f0)
    for delta in [0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0]:
        r = D.fit_tcell(t, f, f0, fixed_delta=delta)
        prof_rows.append({"tissue": tissue, "delta": delta, "rss": 2 * r.cost, "rss_free": 2 * full.cost, "mice": len(d)})
prof = pd.DataFrame(prof_rows)
best = np.minimum(prof.groupby("tissue")["rss"].transform("min"), prof["rss_free"])   # best fit found anywhere
prof["rss_ratio"] = prof["rss"] / best
prof.round(4).to_csv(A.figure("dynamics_tcell_profile.csv"), index=False)
for tissue, (full, f0) in fits.items():
    th, dl, k = full.x
    print(f"{tissue}: theta = {th:.3f}, delta = {dl:.3f}/month (thymic half-life {np.log(2) / max(dl, 1e-9):.0f} months), "
          f"k = {k:.3f}/month, f0 = {f0:.2f}")
prof.pivot(index="delta", columns="tissue", values="rss_ratio").round(3)

### 6. (7e) Forecasting a held-out age, bulk

Per organ, sex and plate effects are first removed from every gene (fitted on the training mice only), because a held-out age with a different sex mix would otherwise shift for a reason unrelated to aging. The 2,000 most variable genes **of the training mice** (the held-out mice do not take part in choosing them) are then summarized by **3 programs** (principal components, standardized), fitted **without** the held-out age so it cannot leak in. Each of the 8 interior ages (3–24 months) is held out in turn (interpolation), and 27 months is predicted from 1–24 (extrapolation). Each model predicts the held-out age's mean program vector; error = squared distance to the real mean. **Skill** = 1 − error / error of the nearest-age baseline: above 0 beats simply using the closest age. Because the held-out age is also a held-out set of mice and libraries, beating the baseline requires real dynamics, not memorized samples. The neural ODE makes this cell slow (~30 min).

Note: the bulk ages are evenly spaced (3 months apart from 3 to 27), so for an interior age the nearest-age baseline (the mean of the two equally near neighbours) and linear interpolation give the same prediction.

**Seeds and tests.** A neural network depends on its random start. The one claim that rests on the neural ODE (extrapolation) is therefore fitted from 5 seeds and the median error over seeds is used; interpolation uses one seed. Each model's skills are then tested against 0 across organs with a two-sided sign test (ties dropped) and a Wilcoxon signed-rank test, Benjamini–Hochberg corrected over the six models within interpolation and within extrapolation.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
NODE_SEEDS = [0, 1, 2, 3, 4]       # extrapolation only: the one claim that rests on the neural ODE
fc = []
for organ, idx in bobs.groupby("Organ").indices.items():
    sub = bulk[idx]
    X_all = sub.X.toarray() if hasattr(sub.X, "toarray") else np.asarray(sub.X)
    age = sub.obs["Age"].to_numpy(float)
    ages = np.sort(np.unique(age))
    cov = sub.obs[["Sex", "Plate"]].reset_index(drop=True)
    for hold in ages[1:]:
        tr = age != hold
        X = X_all[:, np.argsort(X_all[tr].var(axis=0))[::-1][:2000]]    # chosen without the held-out mice
        Xr = D.residualize(X, cov, tr)
        pca = PCA(3, random_state=0).fit(Xr[tr])
        sc_ = StandardScaler().fit(pca.transform(Xr[tr]))
        P = sc_.transform(pca.transform(Xr))
        T = np.array([a for a in ages if a != hold])
        Y = np.array([P[age == a].mean(axis=0) for a in T])
        truth = P[age == hold].mean(axis=0)
        kind = "extrapolate" if hold == ages.max() else "interpolate"
        for m in D.FORECASTERS:
            for seed in (NODE_SEEDS if (m == "neural_ode" and kind == "extrapolate") else [0]):
                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                    try:
                        pred = D.forecast(T, Y, hold, m, data_t=age[tr], data_y=P[tr], seed=seed)
                    except Exception:
                        pred = np.full(3, np.nan)
                fc.append({"organ": organ, "hold": hold, "kind": kind, "model": m, "seed": seed,
                           "error": float(np.sum((pred - truth) ** 2))})
fc_seeds = pd.DataFrame(fc)
fc = (fc_seeds.groupby(["organ", "hold", "kind", "model"], as_index=False)
              .agg(error=("error", "median"), seeds=("seed", "size")))      # median over seeds; NaN only if all failed
base = fc[fc["model"] == "nearest"].set_index(["organ", "hold"])["error"]
fc["skill"] = 1 - fc["error"] / fc.set_index(["organ", "hold"]).index.map(base).to_numpy()
fc.round(4).to_csv(A.figure("dynamics_forecast_bulk.csv"), index=False)
node = fc_seeds[(fc_seeds["model"] == "neural_ode") & (fc_seeds["kind"] == "extrapolate")].copy()
node["skill"] = 1 - node["error"] / node.set_index(["organ", "hold"]).index.map(base).to_numpy()
print("neural ODE, extrapolation, median skill per seed:", node.groupby("seed")["skill"].median().round(2).to_dict())

In [ ]:
from scipy.stats import binomtest, wilcoxon
from statsmodels.stats.multitest import multipletests
summ = fc.groupby(["kind", "model"]).agg(cases=("skill", "size"), median_skill=("skill", "median"),
                                         beats_nearest=("skill", lambda v: (v > 0).mean()),
                                         failed=("error", lambda v: v.isna().sum())).round(3)
lin = fc[fc["model"] == "linear"].set_index(["organ", "hold"])["error"]
fc["beats_linear"] = fc["error"] < fc.set_index(["organ", "hold"]).index.map(lin).to_numpy()
summ["beats_linear"] = fc.groupby(["kind", "model"])["beats_linear"].mean().round(3)
tests = []
for (kind, m), g in fc[fc["model"] != "nearest"].groupby(["kind", "model"]):
    s = g["skill"].dropna()
    s = s[~np.isclose(s, 0)]                       # ties with the baseline carry no direction
    k = int((s > 0).sum())
    tests.append({"kind": kind, "model": m, "wins": f"{k}/{len(s)}",
                  "sign_p": binomtest(k, len(s)).pvalue if len(s) else np.nan,
                  "wilcoxon_p": wilcoxon(s).pvalue if len(s) >= 6 else np.nan})
tests = pd.DataFrame(tests)
for kind in tests["kind"].unique():
    sel = (tests["kind"] == kind) & tests["sign_p"].notna()
    tests.loc[sel, "sign_p_bh"] = multipletests(tests.loc[sel, "sign_p"], method="fdr_bh")[1]
    sel = (tests["kind"] == kind) & tests["wilcoxon_p"].notna()
    tests.loc[sel, "wilcoxon_p_bh"] = multipletests(tests.loc[sel, "wilcoxon_p"], method="fdr_bh")[1]
summ = summ.join(tests.set_index(["kind", "model"]))
summ.round(3).to_csv(A.figure("dynamics_forecast_summary.csv"))
summ.round(3)

In [ ]:
order = [m for m in D.FORECASTERS if m != "nearest"]
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=fc[fc["model"] != "nearest"], x="skill", y="model", hue="kind", order=order, ax=ax, showfliers=False)
ax.axvline(0, color="gray", lw=0.8)
ax.set_xlim(-3, 1.05)
ax.set_xlabel("skill vs nearest age (1 = perfect, 0 = no better, < 0 = worse)")
fig.savefig(A.figure("dynamics_forecast_bulk.png"), dpi=200, bbox_inches="tight")

### 7. (7e) Forecasting a held-out age, single cells: optimal transport

For whole cell populations, **optimal transport** (Waddington-OT; here `moscot`) couples the cells of one age to the next and interpolates the population in between. The test of Schiebinger et al. 2019: hold out a middle age, interpolate it from its neighbours, and measure the Wasserstein distance (a distance between two clouds of cells) to the real cells. It is compared with a **random** interpolation (same cells, no coupling), with the distance from each neighbour, and with the **batch distance** between mice of the held-out age (the noise floor: how different two mice of the same age already are). Rows: robust rows with ages 18, 21 and 24 months (limb muscle, kidney), 300 cells per mouse, 30 principal components.

**Two caveats fixed in advance.** Only 7 rows have the three ages, so a pattern across them is a count, not a test. And sex is tied to these ages: in limb muscle and kidney the 21m mice are all female while the 24m mice are all male and 18m is mixed, so the interpolated population carries a sex mix the real 21m cells do not; part of every distance here can be sex. The batch floor (two 21m mice, same sex) does not contain it.

In [ ]:
from moscot.problems.time import TemporalProblem
ot = []
for tissue in ["Limb_Muscle", "Kidney"]:
    a = sc.read_h5ad(ag.paths(tissue).processed)
    for ct in atlas.loc[atlas["robust"] & atlas["tissue"].eq(tissue), "cell_type"]:
        sub = a[(a.obs["cell_ontology_class"] == ct).to_numpy() & a.obs["age"].isin(["18m", "21m", "24m"]).to_numpy()]
        keep = sub.obs.groupby("mouse.id", observed=True).sample(frac=1, random_state=0) \
                  .groupby("mouse.id", observed=True).head(300).index
        sub = sub[keep].copy()
        if sub.obs["age"].nunique() < 3 or sub.obs.groupby("age", observed=True)["mouse.id"].nunique().min() < 2:
            continue
        sc.pp.pca(sub, n_comps=30, random_state=0)
        sub.obs["months"] = sub.obs["age"].astype(str).map(ag.AGE_MONTHS).astype(float)
        sub.obs["mouse"] = sub.obs["mouse.id"].astype(str)
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            # policy "triu" solves every pair of ages, including 18 -> 24 directly (21 held out)
            tp = TemporalProblem(sub).prepare(time_key="months", joint_attr="X_pca", policy="triu").solve(epsilon=1e-2)
            interp = tp.compute_interpolated_distance(18.0, 21.0, 24.0, seed=0)
            rand = tp.compute_random_distance(18.0, 21.0, 24.0, seed=0)
            d_src, d_tgt = tp.compute_time_point_distances(18.0, 21.0, 24.0)
            batch = tp.compute_batch_distances(21.0, batch_key="mouse")
        ot.append({"tissue": tissue, "cell_type": ct, "cells": sub.n_obs, "interpolated": interp, "random": rand,
                   "from_18m": d_src, "from_24m": d_tgt, "batch_floor_21m": batch})
    del a
ot = pd.DataFrame(ot)
ot["beats_random"] = ot["interpolated"] < ot["random"]
ot["beats_nearest"] = ot["interpolated"] < ot[["from_18m", "from_24m"]].min(axis=1)
ot.round(3).to_csv(A.figure("dynamics_ot_interpolation.csv"), index=False)
ot.round(3)

### 8. Figures

The classic dynamical-systems views of the models above. Parameters in 8.1 are illustrative (those of the simulation in section 3); everything else is fitted to the data.

#### 8.1 Karin model: rate balance and the loss of a steady state

Left: removal $\beta X/(\kappa + X)$ (curve) against production $\eta t$ (horizontal lines, one per age). Where they cross is a stable steady state $X^* = \kappa \eta t/(\beta - \eta t)$. Removal saturates at $\beta$, so once production passes it (here at $t = \beta/\eta = 25$ months) the lines no longer cross: the steady state disappears and $X$ runs away. Middle: $X^*$ against age diverges at that point. Right: the stochastic model's individuals over age. Individuals lag behind the moving fixed point, so within 30 months they curve upward (accelerate) rather than visibly explode.

In [ ]:
eta, beta, kappa = 0.02, 0.5, 1.0
X = np.linspace(0, 30, 600)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
ax = axes[0]
ax.plot(X, beta * X / (kappa + X), color="k", lw=2, label="removal  βX/(κ+X)")
ax.axhline(beta, color="k", ls=":", lw=1)
for t, col in zip([6, 12, 18, 24, 30], sns.color_palette("viridis", 5)):
    p = eta * t
    ax.axhline(p, color=col, lw=1.5, label=f"production at {t} m")
    if p < beta:
        x_star = kappa * p / (beta - p)
        ax.plot(x_star, p, "o", color=col, ms=7, mec="k")
ax.set_xlabel("X (senescent burden)"); ax.set_ylabel("rate"); ax.set_ylim(0, 0.7)
ax.set_title("fixed point = crossing; none once ηt > β"); ax.legend(fontsize=7, loc="lower right")
t = np.linspace(0.5, 24.9, 300)
axes[1].plot(t, kappa * eta * t / (beta - eta * t), color="k")
axes[1].axvline(beta / eta, color="r", ls="--", lw=1)
axes[1].set_ylim(0, 30); axes[1].set_xlabel("age (months)"); axes[1].set_ylabel("steady state X*")
axes[1].set_title(f"X* diverges at t = β/η = {beta / eta:.0f} months")
for q in [5, 25, 50, 75, 95]:
    axes[2].plot(ts, np.percentile(xs, q, axis=0), color="k", lw=2 if q == 50 else 0.8)
axes[2].axvline(beta / eta, color="r", ls="--", lw=1)
axes[2].set_xlabel("age (months)"); axes[2].set_ylabel("X"); axes[2].set_title("2,000 simulated individuals (5–95th pct)")
fig.tight_layout()
fig.savefig(A.figure("dynamics_karin_phase.png"), dpi=200, bbox_inches="tight")

#### 8.2 p16 per mouse, with the three growth models

In [ ]:
organs = [s.split(", ")[1] for s in growth["series"] if s.startswith("p16")]
fig, axes = plt.subplots(3, 4, figsize=(14, 9), sharex=True)
tt = np.linspace(1, 27, 200)
for ax, organ in zip(axes.flat, organs):
    idx = bobs.index[bobs["Organ"] == organ]
    t_, y_ = bobs.loc[idx, "Age"].to_numpy(float), p16.loc[idx].to_numpy()
    fit = D.fit_growth(t_, y_)
    ax.scatter(t_, y_, s=12, color="gray", alpha=0.7)
    ax.plot(tt, np.polyval(fit["linear"]["params"], tt), label="linear")
    if fit["exponential"]["params"] is not None:
        a_, c_, r_ = fit["exponential"]["params"]
        ax.plot(tt, a_ + c_ * (np.exp(r_ * tt) - 1), label="exponential")
    q = fit["karin"]["params"]
    ax.plot(tt, q[0] + D.karin_mean(tt, *q[1:]), label="Karin")
    best = min(fit, key=lambda k: fit[k]["aic"])
    ax.set_title(f"{organ} (best: {best})", fontsize=9)
for ax in axes.flat[len(organs):]:
    ax.axis("off")
axes.flat[0].legend(fontsize=7)
fig.supxlabel("age (months)"); fig.supylabel("p16 (Cdkn2a), log CPM")
fig.tight_layout()
fig.savefig(A.figure("dynamics_p16_fits.png"), dpi=200, bbox_inches="tight")

#### 8.3 Phase portrait: where an organ's state flows with age

Sex and plate effects are removed from every gene. To avoid a circular picture, the mice of each organ are split in two halves, alternating within each age: the 500 genes most correlated with age, and the two programs (PC1, PC2), are chosen on **one half**, and only the **other half** is drawn. Choosing genes by age and then showing that the same mice line up by age would be guaranteed by construction; on the other half it is not. Dots are held-out mice coloured by age; the black path joins their per-age means from 1 to 27 months; the grey streamlines are the vector field $\dot{x} = Ax + b$ of a linear ODE fitted to that path. This figure illustrates the trajectory; the forecasting test of section 6 is the evidence.

In [ ]:
from sklearn.decomposition import PCA
def programs(organ, k=2):
    """Genes and programs chosen on one half of the mice (alternating within each age);
    returns the other half only, so the age ordering shown is not built in."""
    idx = bobs.groupby("Organ").indices[organ]
    sub = bulk[idx]
    Xo = sub.X.toarray() if hasattr(sub.X, "toarray") else np.asarray(sub.X)
    Xo = Xo[:, np.argsort(Xo.var(axis=0))[::-1][:5000]]
    Xo = D.residualize(Xo, sub.obs[["Sex", "Plate"]].reset_index(drop=True), np.ones(len(Xo), bool))
    age = sub.obs["Age"].to_numpy(float)
    o = pd.DataFrame({"age": age, "mouse": sub.obs["MouseID"].astype(str).to_numpy()}).sort_values(["age", "mouse"])
    pick = np.zeros(len(age), bool)
    pick[o.index[o.groupby("age").cumcount() % 2 == 0]] = True          # selection half
    rho = pd.DataFrame(Xo[pick]).rank().corrwith(pd.Series(age[pick]).rank()).abs().fillna(0).to_numpy()
    top = np.argsort(rho)[::-1][:500]                                  # 500 genes most correlated with age, selection half
    pca = PCA(k, random_state=0).fit(Xo[pick][:, top])
    P = pca.transform(Xo[~pick][:, top])
    P = (P - P.mean(axis=0)) / P.std(axis=0)
    age = age[~pick]
    T = np.sort(np.unique(age))
    rho_shown = [spearmanr(age, P[:, j])[0] for j in range(k)]
    print(f"{organ}: {(~pick).sum()} held-out mice; |Spearman rho| of programs with age: {np.round(np.abs(rho_shown), 2)}")
    return P, age, T, np.array([P[age == a].mean(axis=0) for a in T])

def linear_field(T, Y):
    dY = np.diff(Y, axis=0) / np.diff(T)[:, None]
    Xm = (Y[1:] + Y[:-1]) / 2
    Dm = np.hstack([Xm, np.ones((len(Xm), 1))])
    W = np.linalg.solve(Dm.T @ Dm + 1e-2 * np.eye(Dm.shape[1]), Dm.T @ dY)
    return W[:-1].T, W[-1]

show = ["Kidney", "Lung", "Limb_Muscle", "Mesenteric_Fat"]
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
for ax, organ in zip(axes, show):
    P, age, T, Y = programs(organ)
    Am, bm = linear_field(T, Y)
    g = np.linspace(P.min() - 0.3, P.max() + 0.3, 25)
    GX, GY = np.meshgrid(g, g)
    U = Am[0, 0] * GX + Am[0, 1] * GY + bm[0]
    V = Am[1, 0] * GX + Am[1, 1] * GY + bm[1]
    ax.streamplot(GX, GY, U, V, color="lightgray", density=1.0, linewidth=0.8)
    s = ax.scatter(P[:, 0], P[:, 1], c=age, cmap="viridis", s=18, edgecolor="none")
    ax.plot(Y[:, 0], Y[:, 1], "-o", color="k", ms=4, lw=1.5)
    ax.annotate("1 m", Y[0], fontsize=8); ax.annotate("27 m", Y[-1], fontsize=8)
    ax.set_title(organ); ax.set_xlabel("program 1"); ax.set_ylabel("program 2")
fig.colorbar(s, ax=axes, label="age (months)", shrink=0.8)
fig.savefig(A.figure("dynamics_phase_portrait.png"), dpi=200, bbox_inches="tight")

#### 8.4 T-cell compartments: fit, and why the rate cannot be identified

Left: naive fraction per mouse with the best-fitting curve. Middle: lung, the same data fitted three times with the involution rate δ **fixed** 100-fold apart (0.01, 0.1, 1 per month), the other parameters refitted each time: the curves nearly coincide across the sampled ages and differ mainly where there are no mice. Right: the profile likelihood for δ (residual relative to the best fit; 1 = as good as the best). This is what "not identifiable" means: very different biology, indistinguishable data. No rate from this fit is reported as a result.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
tt = np.linspace(0, 31, 200)
for (tissue, d), col in zip(tc.groupby("tissue"), sns.color_palette("Set1", 2)):
    full, f0 = fits[tissue]
    th, dl, k = full.x
    pr = prof[prof["tissue"] == tissue]
    axes[0].scatter(d["months"], d["naive_frac"], color=col, s=30, label=tissue)
    axes[0].plot(tt, D.tcell_fraction(tt, th, dl, k, f0), color=col)
    axes[2].plot(pr["delta"], pr["rss_ratio"], "-o", color=col, label=tissue)
lung = tc[tc["tissue"] == "Lung"]
t_l, f_l = lung["months"].to_numpy(float), lung["naive_frac"].to_numpy(float)
f0_l = fits["Lung"][1]
axes[1].scatter(t_l, f_l, color="k", s=30, zorder=3, label="lung mice")
for delta, ls in zip([0.01, 0.1, 1.0], [":", "--", "-"]):
    r = D.fit_tcell(t_l, f_l, f0_l, fixed_delta=delta)
    th_, k_ = r.x
    axes[1].plot(tt, D.tcell_fraction(tt, th_, delta, k_, f0_l), ls=ls, color="tab:blue",
                 label=f"δ = {delta}/month, residual {2 * r.cost:.3f}")
axes[0].set_xlabel("age (months)"); axes[0].set_ylabel("naive fraction of T cells"); axes[0].set_ylim(0, 1); axes[0].legend()
axes[1].set_xlabel("age (months)"); axes[1].set_ylabel("naive fraction of T cells"); axes[1].set_ylim(0, 1)
axes[1].set_title("Lung: involution rates 100-fold apart"); axes[1].legend(fontsize=7)
axes[2].set_xscale("log"); axes[2].axhline(1, color="gray", lw=0.8)
axes[2].set_xlabel("involution rate δ (per month), fixed"); axes[2].set_ylabel("residual / best residual"); axes[2].legend()
fig.tight_layout()
fig.savefig(A.figure("dynamics_tcell.png"), dpi=200, bbox_inches="tight")

#### 8.5 Forecasting a held-out age, seen in program space

Kidney, two cases: 15 months held out (interpolation) and 27 months held out (extrapolation). Grey: training mice and per-age means; black crosses: the held-out mice; coloured markers: each model's prediction of their mean.

In [ ]:
idx = bobs.groupby("Organ").indices["Kidney"]
sub = bulk[idx]
Xo = sub.X.toarray() if hasattr(sub.X, "toarray") else np.asarray(sub.X)
X_all = Xo
age = sub.obs["Age"].to_numpy(float)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
markers = dict(zip(D.FORECASTERS, ["s", "D", "^", "v", "P", "X", "*"]))
for ax, hold in zip(axes, [15.0, 27.0]):
    tr = age != hold
    Xo = X_all[:, np.argsort(X_all[tr].var(axis=0))[::-1][:2000]]    # chosen without the held-out mice
    Xr = D.residualize(Xo, sub.obs[["Sex", "Plate"]].reset_index(drop=True), tr)
    pca = PCA(2, random_state=0).fit(Xr[tr])
    Pk = pca.transform(Xr)
    mu_, sd_ = Pk[tr].mean(axis=0), Pk[tr].std(axis=0)
    Pk = (Pk - mu_) / sd_
    T = np.sort(np.unique(age[tr]))
    Y = np.array([Pk[age == a].mean(axis=0) for a in T])
    ax.scatter(Pk[tr, 0], Pk[tr, 1], color="lightgray", s=12)
    ax.plot(Y[:, 0], Y[:, 1], "-o", color="gray", ms=3)
    ax.scatter(Pk[~tr, 0], Pk[~tr, 1], color="k", marker="x", s=40, label=f"held-out {hold:.0f} m mice")
    for m, col in zip(D.FORECASTERS, sns.color_palette("tab10", len(D.FORECASTERS))):
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            pr_ = D.forecast(T, Y, hold, m, data_t=age[tr], data_y=Pk[tr])
        if not np.all(np.isfinite(pr_)):
            print(f"{m} failed for {hold:.0f} months (solution diverged)")
            continue
        ax.scatter(*pr_[:2], marker=markers[m], s=80, color=col, edgecolor="k", label=m)
    ax.set_title(f"Kidney, {'interpolate' if hold < 27 else 'extrapolate'} {hold:.0f} months")
    ax.set_xlabel("program 1"); ax.set_ylabel("program 2")
axes[1].legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.savefig(A.figure("dynamics_forecast_example.png"), dpi=200, bbox_inches="tight")

#### 8.6 Optimal transport: interpolating 21-month cells from 18 and 24 months

In [ ]:
ot_plot = pd.read_csv(A.figure("dynamics_ot_interpolation.csv"))
ot_plot["row"] = ot_plot["tissue"] + "\n" + ot_plot["cell_type"].str.slice(0, 26)
long = ot_plot.melt(id_vars="row", value_vars=["interpolated", "random", "from_18m", "from_24m", "batch_floor_21m"],
                    var_name="comparison", value_name="Wasserstein distance to real 21-month cells")
fig, ax = plt.subplots(figsize=(10, 0.6 * len(ot_plot) + 2))
sns.barplot(data=long, y="row", x="Wasserstein distance to real 21-month cells", hue="comparison", ax=ax)
ax.set_ylabel(""); ax.legend(fontsize=8, title="")
fig.savefig(A.figure("dynamics_ot.png"), dpi=200, bbox_inches="tight")

### Findings

Exploratory throughout. Bulk: 17 organs, 56 mice, 10 ages (sex and plate removed before forecasting). Single cell: 26 robust rows (7b), spleen and lung T cells (7c), 7 muscle and kidney rows (optimal transport). Fixes made before results were written: a stiff-solver hang in the Karin fit (`LSODA`), a two-pool T-cell model that collapsed to numerical noise (replaced by the homeostatic fraction model), a SINDy fallback that silently returned a baseline (now counted as a failure), and sex dominating the kidney programs (now regressed out). Fixes from the critical review: forecast genes chosen on training mice only, 5 neural-ODE seeds with sign and Wilcoxon tests, a split-half phase portrait, the SINDy equation figure removed, the T-cell figure redrawn to show non-identifiability.

- **7a. Acceleration cannot be tested with this readout.** For p16 a straight line wins by AIC in 10 of 11 organs (marrow: exponential, by 0.1); the Karin saturating-removal model never wins (2.4–6.2 AIC behind), nor for the body-wide MOFA aging factor (14.7 behind). Bulk p16 sits near the detection floor (0–0.3 log CPM, many zeros), so this says the data show no curvature, not that aging is linear. The spread between mice grows with age in 8 of 12 series (median ρ 0.16): consistent with the stochastic model, but almost any model of individuals with different rates predicts the same, so it is not evidence for this one.
- **7b. Heterogeneity rises; no early-warning signal.** Cell-to-cell distance within a mouse (depth-matched) rises with age in 16 of 26 robust rows, 10 of them stable with any one age group left out (bladder endothelium ρ 0.89, lung alveolar macrophage 0.79, muscle T cell 0.73, kidney macrophage 0.67, lung fibroblast 0.62). The leading-eigenvalue share and gene–gene correlation show no direction (6–8 stable rises, 6–8 stable falls). Each mouse is seen once and each age is its own batch, so this is a description of aged populations, not a test of critical slowing down.
- **7c. The thymic involution rate is not identifiable from these ages.** Fits with δ fixed 100-fold apart (0.01, 0.1, 1 per month) nearly coincide over the sampled ages (figure 8.4), and the profile likelihood varies by only 1–13% across δ = 0.003–3 per month. No rate is reported.
- **7e. Forecasting a held-out age: nothing beats the nearest age.** Inside the range, sigmoid wins 74 of 136 cases (sign p 0.35), spline 61, linear interpolation equals the nearest age except at uneven spacing (11 of 17 there, p 0.33); the neural ODE (57 of 136), the linear ODE (33) and SINDy (20 of the 86 that ran; 50 diverged) are worse (BH-corrected Wilcoxon ≤ 0.001). **Beyond the range (27 months from 1–24), no model beats carrying the 24-month value forward:** the neural ODE wins 8 of 17 organs (sign p 1.0; median skill −0.27, from −0.37 to 0.29 depending on the seed), sigmoid 5, spline and linear ODE 3, linear extrapolation 1; SINDy diverges in 10 of 17. An earlier single-seed run, with genes chosen on all mice, had the neural ODE ahead (median skill 0.21); that was one seed and is withdrawn.
- **Phase portrait (8.3).** With genes and programs chosen on one half of the mice, the other half still orders by age (|ρ| of program 1 with age: kidney 0.92, limb muscle 0.83, mesenteric fat 0.79; lung 0.58, with program 2 at 0.76), so the steady path is not an artifact of choosing age genes. It remains an illustration; the forecasting test is the evidence, and it finds no dynamics beyond "stay near the neighbouring age".
- **7e. Optimal transport, 2 of 7 rows.** Interpolating 21-month cells from 18 and 24 months beats a random interpolation, the nearest age and the distance between two 21-month mice for muscle endothelium (54.9 vs random 65.1, floor 58.6) and muscle mesenchymal stem cells (55.7 vs 63.8, floor 63.4); for muscle T, B, macrophage and satellite cells and kidney macrophage a random interpolation does as well. The 21-month mice are all female and their neighbours mostly male, so part of every distance is sex; with 7 rows this is a count, not a test.